In [1]:
"""
Этап 7. Список кандидатов на сокращение.

Начинаем с группы CZ (2426 SKU). Применяем дополнительные фильтры:
- возраст товара;
- клиентская база;
- повторные покупки;
- сезонность.

Цель — отобрать безопасные кандидаты для ~20% сокращения.
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from scripts.db_config import read_sql, execute_sql, _get_conn_params

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

REPORTS_DIR = ROOT_DIR / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

print("✅ Окружение готово")

✅ Окружение готово


In [2]:
# Базовая таблица
products = read_sql("""
    SELECT * FROM abc_xyz
    WHERE abc_xyz IN ('CZ', 'CY', 'BZ', 'BY');
""")
products["revenue"] = products["revenue"].astype(float)
products["cv"] = products["cv"].astype(float)

# Доп. признаки из sales
extra = read_sql("""
    SELECT
        stock_code,
        MIN(invoice_date) AS first_sale,
        MAX(invoice_date) AS last_sale,
        COUNT(DISTINCT customer_id) FILTER (
            WHERE customer_id IS NOT NULL
        ) AS unique_customers,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT invoice_no) AS unique_invoices
    FROM sales
    GROUP BY stock_code;
""")
extra["first_sale"] = pd.to_datetime(extra["first_sale"])
extra["last_sale"] = pd.to_datetime(extra["last_sale"])

# Количество клиентов с повторными покупками
repeat = read_sql("""
    WITH cust_purchases AS (
        SELECT stock_code, customer_id, COUNT(*) AS n
        FROM sales
        WHERE customer_id IS NOT NULL
        GROUP BY stock_code, customer_id
    )
    SELECT
        stock_code,
        COUNT(*) FILTER (WHERE n >= 2) AS repeat_customers
    FROM cust_purchases
    GROUP BY stock_code;
""")

# Возраст товара в месяцах
reference_date = pd.Timestamp("2011-11-30")
extra["age_months"] = (
    (reference_date - extra["first_sale"]).dt.days / 30.44
).round(1)

products = products.merge(extra, on="stock_code", how="left")
products = products.merge(repeat, on="stock_code", how="left")

print(f"Загружено SKU из групп CZ/CY/BZ/BY: {len(products):,}")
display(products.head())

Загружено SKU из групп CZ/CY/BZ/BY: 3,843


,stock_code,description,abc_category,xyz_category,abc_xyz,revenue,revenue_share,total_quantity,orders,customers,active_months,mean_qty,std_qty,cv,first_sale,last_sale,unique_customers,total_rows,unique_invoices,age_months,repeat_customers
0,22445,PENCIL CASE LIFE IS BEAUTIFUL,B,Y,BY,"4,052.12",0.00020150744468491548,1321,275,147,22,54.625,41.20963934566766,0.75,2010-03-17 15:24:00,2011-12-08 14:39:00,147,298,275,20.40,40.00
1,20982,12 PENCILS TALL TUBE SKULLS,B,Y,BY,"4,309.25",0.00021429423511852365,4928,576,316,25,201.04166666666666,152.6822624403008,0.76,2009-12-01 11:37:00,2011-12-09 10:03:00,316,608,576,23.90,97.00
2,22285,HANGING HEN ON NEST DECORATION,B,Z,BZ,"4,307.53",0.00021420870142138286,2739,261,165,25,113.58333333333333,157.93878473073744,1.39,2009-12-06 13:10:00,2011-12-05 17:24:00,165,278,261,23.80,39.00
3,21245,GREEN POLKADOT PLATE,B,Y,BY,"4,305.54",0.00021410974092294675,2589,304,146,25,103.16666666666667,59.56835476063519,0.58,2009-12-01 11:21:00,2011-12-07 10:12:00,146,316,304,23.90,48.00
4,21153,RED SPOTTY TABLECLOTH,B,Z,BZ,"4,294.69",0.00021357018242644827,393,139,91,10,16.375,28.057697250487255,1.71,2009-12-01 14:47:00,2010-09-22 15:17:00,91,139,139,23.90,15.00


In [3]:
cz = products[products["abc_xyz"] == "CZ"].copy()

print(f"Профиль CZ ({len(cz):,} SKU):")
display(cz[[
    "revenue", "orders", "customers", "unique_customers",
    "repeat_customers", "active_months", "age_months", "cv"
]].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(2))

# Распределения
print("\nРаспределение по числу уникальных клиентов:")
bins = [0, 1, 5, 10, 20, 50, 200]
labels = ["1", "2-5", "6-10", "11-20", "21-50", "51+"]
cz["cust_bin"] = pd.cut(cz["unique_customers"], bins=bins, labels=labels)
display(cz["cust_bin"].value_counts().sort_index().to_frame("SKU"))

print("\nРаспределение по числу активных месяцев:")
bins2 = [0, 1, 3, 6, 12, 18, 25]
labels2 = ["1", "2-3", "4-6", "7-12", "13-18", "19-24"]
cz["active_bin"] = pd.cut(cz["active_months"], bins=bins2, labels=labels2)
display(cz["active_bin"].value_counts().sort_index().to_frame("SKU"))

print("\nРаспределение по возрасту (месяцев):")
bins3 = [0, 3, 6, 12, 18, 25]
labels3 = ["0-3", "4-6", "7-12", "13-18", "19-24"]
cz["age_bin"] = pd.cut(cz["age_months"], bins=bins3, labels=labels3)
display(cz["age_bin"].value_counts().sort_index().to_frame("SKU"))

Профиль CZ (2,426 SKU):


,revenue,orders,customers,unique_customers,repeat_customers,active_months,age_months,cv
count,"2,426.00","2,426.00","2,426.00","2,426.00","2,167.00","2,426.00","2,426.00","2,426.00"
mean,374.04,36.74,20.23,20.23,3.44,9.17,20.31,2.65
std,350.53,40.15,21.70,21.70,4.46,6.72,6.75,1.17
min,0.38,1.00,0.00,0.00,0.00,1.00,0.00,1.00
25%,67.62,8.00,4.00,4.00,0.00,3.00,21.20,1.68
50%,254.60,24.00,13.00,13.00,2.00,8.00,23.60,2.35
75%,630.55,52.00,30.00,30.00,5.00,14.00,23.90,3.44
90%,946.62,88.00,51.50,51.50,9.00,20.00,23.90,4.80
max,"1,206.54",317.00,129.00,129.00,38.00,25.00,23.90,4.80



Распределение по числу уникальных клиентов:


,SKU
cust_bin,
1,129
2-5,362
6-10,330
11-20,480
21-50,608
51+,258



Распределение по числу активных месяцев:


,SKU
active_bin,
1,251
2-3,377
4-6,437
7-12,619
13-18,436
19-24,306



Распределение по возрасту (месяцев):


,SKU
age_bin,
0-3,182
4-6,54
7-12,62
13-18,156
19-24,1970


In [5]:
# Основной сценарий: Умеренный
products = read_sql("SELECT * FROM abc_xyz WHERE abc_xyz IN ('CZ', 'CY');")

# Загружаем доп. признаки
extra = read_sql("""
    SELECT
        stock_code,
        MIN(invoice_date) AS first_sale,
        COUNT(DISTINCT customer_id) FILTER (WHERE customer_id IS NOT NULL) AS unique_customers,
        COUNT(DISTINCT invoice_no) AS orders_total
    FROM sales
    GROUP BY stock_code;
""")
extra["first_sale"] = pd.to_datetime(extra["first_sale"])

repeat = read_sql("""
    WITH cust_purchases AS (
        SELECT stock_code, customer_id, COUNT(*) AS n
        FROM sales
        WHERE customer_id IS NOT NULL
        GROUP BY stock_code, customer_id
    )
    SELECT stock_code,
           COUNT(*) FILTER (WHERE n >= 2) AS repeat_customers
    FROM cust_purchases
    GROUP BY stock_code;
""")

products = products.merge(extra, on="stock_code", how="left")
products = products.merge(repeat, on="stock_code", how="left")

reference_date = pd.Timestamp("2011-11-30")
products["age_months"] = (
    (reference_date - products["first_sale"]).dt.days / 30.44
).round(1)

# МЯГКИЙ фильтр (678 SKU)
candidates = products[
    (products["age_months"] >= 12) &
    (products["unique_customers"] <= 10) &
    (products["repeat_customers"].fillna(0) == 0) &
    (products["active_months"] <= 18)
].copy()

# Чистим NaN
candidates["repeat_customers"] = candidates["repeat_customers"].fillna(0).astype(int)
candidates["unique_customers"] = candidates["unique_customers"].fillna(0).astype(int)
candidates["age_months"] = candidates["age_months"].fillna(0).round(1)
candidates["cv"] = candidates["cv"].astype(float).round(3)

# Сохраняем в БД
execute_sql("DROP TABLE IF EXISTS candidates;")
execute_sql("""
    CREATE TABLE candidates (
        stock_code        text PRIMARY KEY,
        description       text,
        abc_xyz           text,
        revenue           numeric,
        orders            integer,
        unique_customers  integer,
        repeat_customers  integer,
        active_months     integer,
        age_months        numeric,
        cv                numeric
    );
""")

insert_sql = "INSERT INTO candidates VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s);"
rows = [tuple(r) for r in candidates[[
    "stock_code", "description", "abc_xyz", "revenue",
    "orders", "unique_customers", "repeat_customers",
    "active_months", "age_months", "cv",
]].itertuples(index=False, name=None)]

with psycopg2.connect(**_get_conn_params()) as conn:
    with conn.cursor() as cur:
        cur.executemany(insert_sql, rows)
    conn.commit()

print(f"✅ Таблица candidates сохранена ({len(rows):,} строк)")

# Приводим к float
total_sku_count = 4895
total_rev = 20109033.72
cand_rev = float(candidates["revenue"].sum())

print()
print(f"Проверка:")
print(f"  Всего SKU в abc_xyz: {total_sku_count:,}")
print(f"  Кандидатов на вывод: {len(candidates):,} ({len(candidates)/total_sku_count*100:.2f}%)")
print(f"  Их выручка: £{cand_rev:,.2f} ({cand_rev/total_rev*100:.2f}%)")

✅ Таблица candidates сохранена (678 строк)

Проверка:
  Всего SKU в abc_xyz: 4,895
  Кандидатов на вывод: 678 (13.85%)
  Их выручка: £71,516.71 (0.36%)


In [6]:
check = read_sql("""
    SELECT
        COUNT(*) AS n,
        ROUND(SUM(revenue)::numeric, 2) AS rev
    FROM candidates;
""")
print("Проверка таблицы candidates в БД:")
display(check.T)

Проверка таблицы candidates в БД:


,0
n,678
rev,71516.71


## Этап 7. Список кандидатов на сокращение — ВЫВОДЫ (финал)

### Цель
Бизнес-задача: рассмотреть сокращение ~20% SKU (~979 из 4 895).
Начать с CZ (2 426 SKU) — основной зоны для проверки.

### Методология
Применили **умеренный фильтр** к CZ/CY:
- **Возраст** ≥ 12 мес. — исключаем новые товары.
- **Клиентская база** ≤ 10 — исключаем товары с широкой базой.
- **Повторные покупки** = 0 — исключаем товары с возвращающимися клиентами.
- **Активные месяцы** ≤ 18 — исключаем регулярные товары.

### Сценарии

| Сценарий | SKU | % SKU | Выручка | % выручки | Рычаг |
|----------|-----|-------|---------|-----------|-------|
| Осторожный | 425 | 8.68% | £24 645 | 0.12% | 72x |
| **Умеренный (основной)** | **678** | **13.85%** | **£71 517** | **0.36%** | **38x** |
| Агрессивный | 2 283 | 46.64% | £866 355 | 4.31% | 11x |

*Рычаг = %SKU / %выручки — эффективность сокращения.*

### Рекомендация: умеренный сценарий (678 SKU)

**Почему умеренный:**
- **Близок к бизнес-цели** (~14% vs целевые 20%).
- **Минимальная потеря выручки** — £71 517 (0.36%).
- **Рычаг 38x** — теряем £1 на каждые £38 сокращённой выручки.
- Оставляет «буфер» (~300 SKU) — не действуем максимально жёстко.

**Почему не осторожный (425):**
- Не покрывает бизнес-задачу (8.68% < 20%).
- Оставляет много «пограничных» товаров.

**Почему не агрессивный (2 283):**
- Потеря выручки в 12 раз больше (4.31% vs 0.36%).
- Требует ручной проверки каждой позиции.

### Ключевые выводы
1. **Безопасно сократить 14%** — не 20% (компромисс).
2. **Целевая группа — «мёртвый груз»:** 1-10 клиентов, без повторных
   покупок, не новые, продавались ≤ 18 месяцев.
3. **Доля выручки 0.36%** — потеря минимальна, но снижает нагрузку на склад.

### Ограничения
- **0.36% выручки — историческая.** Реальная потеря может быть меньше
  (клиенты купят аналоги) или больше (потеря клиентов).
- **Не автоматически.** 678 SKU — кандидаты, требуют ручной проверки.
- **Нет данных о марже.** Часть SKU могут быть высокомаржинальными.

### Что дальше
**Этап 8 (Оценка эффекта):** количественная оценка экономии и рисков.